In [48]:
!pip install pandas matplotlib seaborn plotly


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: C:\Users\USER\AppData\Local\Python\pythoncore-3.11-64\python.exe -m pip install --upgrade pip


In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

In [50]:
orders = pd.read_csv('../data/olist_orders_dataset.csv')
orders_reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
oredrs_items = pd.read_csv('../data/olist_order_items_dataset.csv')
customer = pd.read_csv('../data/olist_customers_dataset.csv')

In [51]:
orders_df = pd.DataFrame(orders)
# orders_df.head()
print(orders_df.isna().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [52]:
orders_reviews_df = pd.DataFrame(orders_reviews)
# orders_reviews_df.head()
# orders_reviews_df.shape
print(orders_reviews_df.isna().sum())


review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


In [53]:
orders_items_df = pd.DataFrame(oredrs_items)
# oredrs_items_df.head()
# oredrs_items_df.shape

In [54]:
customer_df = pd.DataFrame(customer)
# customer_df.head()
# customer_df.shape
print(customer_df.isna().sum())


customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


In [55]:
orders_df.groupby('order_status').size()

order_status
approved           2
canceled         625
created            5
delivered      96478
invoiced         314
processing       301
shipped         1107
unavailable      609
dtype: int64

In [56]:
df_filtered = orders_df.loc[orders_df['order_delivered_customer_date'].isna(), ['order_status']]
print(df_filtered.value_counts('order_status'))

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [57]:
df_filtered = orders_df.loc[orders_df['order_status'] == 'unavailable']
print(df_filtered.value_counts('order_delivered_customer_date'))

Series([], Name: count, dtype: int64)


In [58]:
orders_df[orders_df['order_status'] == 'unavailable']['order_delivered_customer_date'].value_counts(dropna=False)

order_delivered_customer_date
NaN    609
Name: count, dtype: int64

In [59]:
orders_clean = orders_df[orders_df['order_status'] == 'delivered']

In [60]:
orders_clean.shape

(96478, 8)

In [61]:
orders_clean.groupby('order_status').size()

order_status
delivered    96478
dtype: int64

In [62]:
orders_clean = pd.merge(orders_clean, customer_df[['customer_id', 'customer_unique_id']], on='customer_id', how='left')

In [63]:
orders_clean.shape
orders_clean.loc[orders_clean['customer_unique_id'].isna(), 'customer_unique_id']

Series([], Name: customer_unique_id, dtype: str)

In [64]:
orders_clean.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
customer_unique_id               str
dtype: object

In [65]:
cols_to_convert = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date','order_estimated_delivery_date']

for col in cols_to_convert:
    orders_clean[col] = pd.to_datetime(orders_clean[col])


In [66]:
order_items_agg = orders_items_df.groupby('order_id')[['price', 'freight_value']].sum()
order_items_agg.head()

,price,freight_value
order_id,,
00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,239.90,19.93
000229ec398224ef6ca0657da4fc703e,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14


In [67]:
order_items_agg['total_value'] = order_items_agg['price'] + order_items_agg['freight_value']
order_items_agg.head()

,price,freight_value,total_value
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


In [68]:
orders_clean = pd.merge(orders_clean, order_items_agg,  left_on='order_id', right_index=True, how='left')
orders_clean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,price,freight_value,total_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,19.90,8.72,28.62


In [69]:
orders_clean.shape


(96478, 12)